In [ ]:
set -euo pipefail

# Paths and namespace
export REPO_ROOT="${REPO_ROOT:-$HOME/recipe-scraper-mlops}"
export WORKFLOW_FILE="${WORKFLOW_FILE:-$REPO_ROOT/devops/workflows/manual-training-workflow.yaml}"
export WORKFLOW_NAMESPACE="${WORKFLOW_NAMESPACE:-recipe-scraper-platform}"

# Argo CLI submit options
export ARGO_SUBMIT_FLAGS="${ARGO_SUBMIT_FLAGS:---watch --log}"

# Parameters passed to the training workflow (defaults from manual-training-workflow.yaml)
export TRAINING_IMAGE="${TRAINING_IMAGE:-192.168.1.11:5000/recipe-scraper-training:latest}"
export TRAINING_MODE="${TRAINING_MODE:-train}"
export TRAINING_CONFIG="${TRAINING_CONFIG:-/app/config/config.yaml}"
export PVC_NAME="${PVC_NAME:-data-pvc-gpu}"
export DATA_DIR="${DATA_DIR:-/data}"
export NUM_PROCESSES="${NUM_PROCESSES:-1}"
export TRAIN_JSONL_PATH="${TRAIN_JSONL_PATH:-/data/training/train.jsonl}"
export EVAL_JSONL_PATH="${EVAL_JSONL_PATH:-/data/training/eval.jsonl}"
export CHECKPOINT_DIR="${CHECKPOINT_DIR:-/data/checkpoints/recipe-correction}"
export HF_CACHE_DIR="${HF_CACHE_DIR:-/data/checkpoints/huggingface-cache}"
export TRAIN_EXTRA_ARGS="${TRAIN_EXTRA_ARGS:---experiment-name recipe-training-manual}"
export MLFLOW_TRACKING_URI="${MLFLOW_TRACKING_URI:-http://192.168.1.11:30800/}"
export MLFLOW_ARTIFACT_SECRET="${MLFLOW_ARTIFACT_SECRET:-mlflow-s3-credentials}"
export CHECK_ARTIFACT_STORE="${CHECK_ARTIFACT_STORE:-true}"

# Build --parameter flags exactly as passed to argo submit
PARAM_FLAGS=(
  --parameter "training-image=${TRAINING_IMAGE}"
  --parameter "training-mode=${TRAINING_MODE}"
  --parameter "training-config=${TRAINING_CONFIG}"
  --parameter "pvc-name=${PVC_NAME}"
  --parameter "data-dir=${DATA_DIR}"
  --parameter "num-processes=${NUM_PROCESSES}"
  --parameter "train-jsonl-path=${TRAIN_JSONL_PATH}"
  --parameter "eval-jsonl-path=${EVAL_JSONL_PATH}"
  --parameter "checkpoint-dir=${CHECKPOINT_DIR}"
  --parameter "hf-cache-dir=${HF_CACHE_DIR}"
  --parameter "train-extra-args=${TRAIN_EXTRA_ARGS}"
  --parameter "mlflow-tracking-uri=${MLFLOW_TRACKING_URI}"
  --parameter "mlflow-artifact-secret=${MLFLOW_ARTIFACT_SECRET}"
  --parameter "check-artifact-store=${CHECK_ARTIFACT_STORE}"
)

echo "Workflow file: $WORKFLOW_FILE"
echo "Namespace: $WORKFLOW_NAMESPACE"
echo "Submit flags: $ARGO_SUBMIT_FLAGS"
echo "Parameter flags:"
printf '  %q\n' "${PARAM_FLAGS[@]}"


# Notebook 3: Manual Training Argo Workflow (CLI)

Use this notebook to submit one-off training workflows from the command line with the same parameter surface as `devops/workflows/manual-training-workflow.yaml`.

## 1) Preflight checks

In [ ]:
set -euo pipefail

command -v argo >/dev/null
command -v kubectl >/dev/null

argo version --client
kubectl config current-context
kubectl get ns "$WORKFLOW_NAMESPACE"
kubectl -n "$WORKFLOW_NAMESPACE" get workflowtemplate recipe-model-training

echo "Preflight checks passed."


## 2) Submit a manual training workflow

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"

ARGO_SUBMIT_CMD=(
  argo submit "$WORKFLOW_FILE"
  --namespace "$WORKFLOW_NAMESPACE"
  ${PARAM_FLAGS[@]}
  $ARGO_SUBMIT_FLAGS
)

echo "Running: ${ARGO_SUBMIT_CMD[*]}"
SUBMIT_OUTPUT="$(${ARGO_SUBMIT_CMD[@]})"
echo "$SUBMIT_OUTPUT"

# Capture generated workflow name (example: recipe-model-training-abcde)
WORKFLOW_NAME="$(printf '%s\n' "$SUBMIT_OUTPUT" | awk '/^Name:/ {print $2}' | tail -n1)"
if [[ -z "$WORKFLOW_NAME" ]]; then
  echo "Could not parse workflow name from submit output." >&2
  exit 1
fi
echo "Submitted workflow: $WORKFLOW_NAME"


## 3) Inspect status and logs

In [ ]:
set -euo pipefail

: "${WORKFLOW_NAME:?Run the submit cell first to populate WORKFLOW_NAME.}"

argo get "$WORKFLOW_NAME" -n "$WORKFLOW_NAMESPACE"
argo logs "$WORKFLOW_NAME" -n "$WORKFLOW_NAMESPACE"
kubectl -n "$WORKFLOW_NAMESPACE" get wf "$WORKFLOW_NAME" -o wide


## 4) Optional: quick parameter overrides

Edit the top cell values, then rerun the submit cell. Examples:

- `TRAIN_EXTRA_ARGS="--experiment-name recipe-training-manual --max-steps 200"`
- `NUM_PROCESSES="2"`
- `TRAIN_JSONL_PATH="/data/training/my_train.jsonl"` and `EVAL_JSONL_PATH="/data/training/my_eval.jsonl"`